## 1. Get Sample Papers for Chunking

In [1]:
import sys
from pathlib import Path
import requests

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"Environment: {sys.executable}")

current_dir = Path.cwd()
if current_dir.name == "tests":
    project_root = current_dir.parent
else:
    project_root = current_dir

print(f"Project Root: {project_root}")

if project_root and (project_root / "src").exists():
    sys.path.insert(0, str(project_root))
else:
    print("Project root not found or src directory missing")
    sys.exit(1)


Python Version: 3.12.11
Environment: /Users/xieqiqi/Learning/LLM/my_first_Rag_project/.venv/bin/python3
Project Root: /Users/xieqiqi/Learning/LLM/my_first_Rag_project


In [2]:
# Get sample papers from database 
from src.db.factory import make_database
from src.models.paper import Paper

print("Fetching sample papers from database")
print("="*40)

database = make_database()

with database.get_session() as session:
    papers = session.query(Paper).filter(
        Paper.raw_text != None,
        Paper.raw_text != ""
    ).limit(3).all()

    if papers:
        print(f"Found {len(papers)} papers with processed text: \n")
        sample_papers = []

        for i, paper in enumerate(papers, 1):
            print(f"{i}.[{paper.arxiv_id}] {paper.title[:60]}...")
            print(f"  Text length: {len(paper.raw_text):,}characters")
            print(f"  Sections available: {'Yes' if paper.sections else 'No'}")

            sample_papers.append({
                "arxiv_id": paper.arxiv_id,
                "title": paper.title,
                "raw_text": paper.raw_text,
                "sections": paper.sections,
                "authors": paper.authors,
                "categories": paper.categories,
                "publish_date": paper.published_date
            })
        test_papers = sample_papers[0]
        print(f"Selected paper for analysis: {test_papers['arxiv_id']}")
    else:
        print("No papers with processed text found.")
        print("Please run the Airflow DAG 'arxiv_paper_ingestion' first.")
        test_paper = None
        sample_papers = []



Fetching sample papers from database
Found 3 papers with processed text: 

1.[2601.08690v1] All Required, In Order: Phase-Level Evaluation for AI-Human ...
  Text length: 63,986characters
  Sections available: Yes
2.[2601.08673v1] Why AI Alignment Failure Is Structural: Learned Human Intera...
  Text length: 74,962characters
  Sections available: Yes
3.[2601.08620v1] ViDoRe V3: A Comprehensive Evaluation of Retrieval Augmented...
  Text length: 104,535characters
  Sections available: Yes
Selected paper for analysis: 2601.08690v1


## 2. Section-based Chunking implementation

In [9]:
import re

def section_based_chunking(text: str, sections_data=None, target_words: int = 600, overlap_words: int=100):
    chunks=[]

    if not sections_data:
        # 
        paragraphs = re.split(r'\n\s*\n', text.strip())
        paragraphs = [p.strip() for p in paragraphs if p.strip()]

        current_chunk = ""
        chunk_index = 0

        for para in paragraphs:
            combined_text = current_chunk + " " + para if current_chunk else para
            if len(combined_text.split()) <= target_words:
                current_chunk = combined_text
            else:
                if current_chunk:
                    chunks.append({
                        'index': chunk_index,
                        'text': current_chunk.strip(),
                        'word_count': len(current_chunk.split()),
                        'section': 'content'
                    })
                    chunk_index += 1
                current_chunk = para

        if current_chunk:
            chunks.append({
                'index': chunk_index,
                'text': current_chunk.strip(),
                'word_count': len(current_chunk.split()),
                'section': 'content'
            })

    else:
        chunk_index = 0

        if isinstance(sections_data, list):
            sections_items = [(item.get('title', f'section_{i}'), item.get('content', '')) for i, item in enumerate(sections_data) if isinstance(item, dict)]
        else:
            sections_items = list(sections_data.items())

        for section_name, section_content in sections_items:
            if not section_content or len(str(section_content).strip()) < 50:
                continue

            section_text = str(section_content).strip()
            words = section_text.split()

            if len(words) <= target_words:
                chunks.append({
                    'index': chunk_index,
                    'text': section_text,
                    'word_count': len(words),
                    'section': section_name
                })
                chunk_index += 1
            else:
                start = 0
                while start < len(words):
                    end = start + target_words
                    chunk_words = words[start:end]
                    chunk_text = ' '.join(chunk_words)

                    chunks.append({
                        'index': chunk_index,
                        'text': chunk_text,
                        'word_count': len(chunk_words),
                        'section': section_name,
                        'has_overlap': start > 0
                    })
                    chunk_index += 1
                    start += (target_words - overlap_words)

                    if end >= len(words):
                        break
    return chunks
        
# Text teh chunking system
if test_papers:
    print("section_based chunking results")
    print("=" * 50)

    chunks = section_based_chunking(
        text=test_papers['raw_text'],
        sections_data=test_papers.get('sections'),
        target_words=600,
        overlap_words=100
    )

    print(f"Paper: {test_papers['arxiv_id']}")
    print(f"Original text: {len(test_papers['raw_text'].split()):,} words")
    print(f"Total chunks created: {len(chunks)}")
    print(f"Average chunk size: {sum(c['word_count'] for c in chunks) / len(chunks):.0f} words")

    print("\nSample chunks:")
    for i in range(min(3, len(chunks))):
        chunk = chunks[i]
        print(f"\nChunk {i+1}: {chunk['section']}")
        print(f"  Words: {chunk['word_count']}")
        print(f"  Text preview: {chunk['text'][:150]}...")

    section_counts = {}
    for chunk in chunks:
        section_counts[chunk['section']] = section_counts.get(chunk['section'], 0) + 1

    print("\nSection distribution:")
    print(f"\nChunks per section (top 5):")
    for section, count in list(section_counts.items())[:5]:
        print(f"  {section}: {count} chunks")
        
else:
    print("No test paper available. Please check database connection.")

    

section_based chunking results
Paper: 2601.08690v1
Original text: 9,335 words
Total chunks created: 30
Average chunk size: 328 words

Sample chunks:

Chunk 1: All Required, In Order: Phase-Level Evaluation for AI-Human Dialogue in Healthcare and Beyond
  Words: 28
  Text preview: Shubham Kulkarni 1 , Alexander Lyzhov 2 , Shiva Chaitanya 1 , Preetam Joshi 2
1 Interactly.ai, California, USA 2 AIMon Labs, California, USA
shubham@i...

Chunk 2: Abstract
  Words: 131
  Text preview: Conversational AI is starting to support real clinical work, but most evaluation methods miss how compliance depends on the full course of a conversat...

Chunk 3: Introduction
  Words: 600
  Text preview: Conversational AI assistants are moving from demonstration prototypes to being deployed in real-world day-to-day clinical work: triage calls, benefits...

Section distribution:

Chunks per section (top 5):
  All Required, In Order: Phase-Level Evaluation for AI-Human Dialogue in Healthcare and Beyond: 1 chunks

## 3. Overlap Strategy Analysis

In [10]:
# Compare different overlap strategies
def compare_overlap_strategies(text: str, sections_data = None):

    overlap_sizes = [0,50,100,150]
    results = []

    print("Comparing overlap strategies")
    print("=" * 50)

    for overlap_size in overlap_sizes:
        chunks = section_based_chunking(
            text=text,
            sections_data=sections_data,
            target_words=600,
            overlap_words=overlap_size
        )

        avg_words = sum(chunk['word_count'] for chunk in chunks) / len(chunks) if chunks else 0

        results.append({
            'overlap_size': overlap_size,
            'chunks': len(chunks),
            'avg_words': avg_words
        })
        print(f"Overlap {overlap_size:3d} words: {len(chunks):3d} chunks, avg {avg_words:.0f} words/chunk")

    print("\nRecommendation: 100-word overlap provides best balance")
    print("- Sufficient context preservation")
    print("- Minimal redundancy")
    print("- Optimal for retrieval accuracy")
    
    return results

if test_papers:
    overlap_results = compare_overlap_strategies(
        text=test_papers['raw_text'],
        sections_data=test_papers.get('sections')
    )
else:
    print("No test paper available. Please check database connection.")


Comparing overlap strategies
Overlap   0 words:  30 chunks, avg 304 words/chunk
Overlap  50 words:  30 chunks, avg 316 words/chunk
Overlap 100 words:  30 chunks, avg 328 words/chunk
Overlap 150 words:  31 chunks, avg 333 words/chunk

Recommendation: 100-word overlap provides best balance
- Sufficient context preservation
- Minimal redundancy
- Optimal for retrieval accuracy


## 4. Standalone Embedding Generation

In [11]:
import httpx
from typing import List
import os
from dotenv import load_dotenv



class JinaEmbeddingGenerator:
    def __init__(self, api_key: str = None, model: str = "jina-embeddings-v3"):
        load_dotenv()
        self.api_key = api_key or os.getenv("JINA_API_KEY")
        self.base_url = "https://api.jina.ai/v1/embeddings"
        self.model = model
        self.embedding_dimention = 1024

        if not self.api_key:
            print("Warning: No Jina API key found. Using dummy embeddings.")

    async def generate_embeddings(self, texts: List[str]) -> List[List[float]]:
        if not self.api_key:
            return [[0.1] * self.embedding_dimention for _ in texts]

        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.api_key}",
        }

        payload = {
            "model": self.model,
            "input": texts,
            "task": "retrieval.passage",
        }

        try:
            async with httpx.AsyncClient() as client:
                response = await client.post(
                    self.base_url,
                    headers=headers,
                    json=payload,
                    timeout=30.0
                )
                response.raise_for_status()
                result = response.json()
                embeddings = [item["embedding"] for item in result["data"]]
                return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            return [[0.1] * self.embedding_dimention for _ in texts]

print("Testing embedding generator...")
print("=" * 50)

embedding_generator = JinaEmbeddingGenerator()

if test_papers and 'chunk' in locals():
    test_texts = [chunk['text'][:500] for chunk in chunks[:3]]
    embeddings = await embedding_generator.generate_embeddings(test_texts)

    print(f"Generated embeddings for {len(embeddings)} chunks")
    print(f"Embedding dimension: {len(embeddings[0])}")

    for i, embedding in enumerate(embeddings):
        norm = sum(x*x for x in embedding)**0.5
        print(f"\nChunk {i+1} embedding:")
        print(f"  Preview: [{embedding[0]:.3f}, {embedding[1]:.3f}, ...]")
        print(f"  Norm: {norm:.3f}")

else:
    test_texts = [
        "Machine learning is a subset of artificial intelligence.",
        "Neural networks are computational models inspired by biology."
    ]
    
    embeddings = await embeddings_generator.generate_embeddings(test_texts)
    print(f"Generated {len(embeddings)} embeddings")
    print(f"Dimension: {len(embeddings[0]) if embeddings else 0}")


Testing embedding generator...
Generated embeddings for 3 chunks
Embedding dimension: 1024

Chunk 1 embedding:
  Preview: [0.108, -0.076, ...]
  Norm: 1.000

Chunk 2 embedding:
  Preview: [0.127, -0.010, ...]
  Norm: 1.000

Chunk 3 embedding:
  Preview: [0.094, -0.071, ...]
  Norm: 1.000


## 5. Unified search system testing

In [12]:
from src.services.opensearch.factory import make_opensearch_client_fresh
from opensearchpy import OpenSearch

print("Unified search system testing")
print("=" * 50)

client = make_opensearch_client_fresh()

client.host = "http://localhost:9200"
client.client = OpenSearch(
    hosts=["http://localhost:9200"],
    use_ssl=False,
    verify_certs=False,
    ssl_show_warn=False
)

stats = client.get_index_stats()
print(f"Index: {stats['index_name']}")
print(f"Documents: {stats['document_count']}")
print(f"Health: {'Healthy' if client.health_check() else 'Unhealthy'}")

if stats['document_count'] > 0:
    print("\n✓ Index contains data. Ready for search testing!")
else:
    print("\n⚠ Index is empty. Please run the Airflow DAG first:")
    print("  1. Open http://localhost:8080 (admin/admin)")
    print("  2. Trigger 'arxiv_paper_ingestion' DAG")

Unified search system testing
Index: arxiv-papers-chunks
Documents: 208
Health: Healthy

✓ Index contains data. Ready for search testing!


## 6. BM25 Keywords search Testing

In [5]:
print("BM25 Keyword search test")
print("=" * 50)

test_queries = [
    "machine learning",
    "neural networks",
    "artificial intelligence"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    try:
        results = client.search_papers(
            query=query,
            size=3
        )
        print(f"  Found: {results.get('total', 0)} results")
        
        for i, hit in enumerate(results.get('hits', [])[:2], 1):
            title = hit.get('title', 'N/A')[:50]
            score = hit.get('score', 0)
            
            print(f"    {i}. {title}... (score: {score:.2f})")
    except Exception as e:
        print(f"Error: {str(e)}")

BM25 Keyword search test

Query: machine learning
  Found: 48 results
    1. APEX-SWE... (score: 2.53)
    2. APEX-SWE... (score: 2.40)

Query: neural networks
  Found: 15 results
    1. APEX-SWE... (score: 3.82)
    2. All Required, In Order: Phase-Level Evaluation for... (score: 6.18)

Query: artificial intelligence
  Found: 42 results
    1. APEX-SWE... (score: 2.32)
    2. All Required, In Order: Phase-Level Evaluation for... (score: 7.41)


## 7. Vector Similarity search testing

In [6]:
print("Vector similarity search test")
print("="*40)

test_queries = [
    "deep learning models",
    "transformer architecture"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    try:
        query_embedding = await embeddings_generator.generate_embeddings([query])
        if query_embedding:
            results = client.search_chunks_vector(
                query_embedding= query_embedding[0],
                size=3
            )
        
            print(f"  Found: {results.get('total', 0)} results")

            for i, hit in enumerate(results.get('hits', [])[:2], 1):
                title = hit.get('title', 'N/A')[:50]
                score = hit.get('score', 0)
                
                print(f"    {i}. {title}... (score: {score:.3f})")
    
    except Exception as e:
        print(f"  Error: {e}")

print("\n✓ Vector search completed!")

Vector similarity search test

Query: 'deep learning models'
  Error: name 'embeddings_generator' is not defined

Query: 'transformer architecture'
  Error: name 'embeddings_generator' is not defined

✓ Vector search completed!


## 8. Hybrid search Testing

In [14]:
print("HYBRID SEARCH TEST (BM25 + VECTOR)")
print("=" * 40)

test_queries = [
    "machine learning algorithms",
    "neural network optimization"
]
try:
    query_embedding = await embedding_generator.generate_embeddings([query])

    for query in test_queries:
        if embedding:
            results = client.search_chunks_hybrid(
                query=query,
                query_embedding=query_embedding[0],
                size=3
            )
            print(f"  Found: {results.get('total', 0)} results")
            print(f"  (60% BM25 + 40% Vector fusion)")
            
            for i, hit in enumerate(results.get('hits', [])[:2], 1):
                title = hit.get('title', 'N/A')[:50]
                score = hit.get('score', 0)
                
                print(f"    {i}. {title}... (hybrid score: {score:.3f})")
except Exception as e:\
    print(f"Error searching chunks: {e}")



HYBRID SEARCH TEST (BM25 + VECTOR)


ERROR:src.services.opensearch.client:Error searching chunks: RequestError(400, 'search_phase_execution_exception', "failed to create query: Field 'embedding' is not knn_vector type.")


Error searching chunks: RequestError(400, 'search_phase_execution_exception', "failed to create query: Field 'embedding' is not knn_vector type.")


## 9. Performance comparison

In [ ]:
import time

print("Search performance comparison")
print("="*50)

query = "machine learning artificial intelligence"
print(f"Test query: {query}")

result_summary = []

start = time.time()
try:
    bm25_results = client.search_papers(query=query, size=5)
    bm25_time = time.time() - start
    results_summary.append({
        'method': 'BM25',
        'time': bm25_time,
        'results': bm25_results
    })
except Exception as e:
    results_summary.append({
        'method': 'BM25',
        'time': None,
        'results': None
    })

start = time.time()
try:
    query_embedding = await embeddings_generator.generate_embeddings([query])
    if query_embedding:
        vector_results = client.search_chunks_vector(
            query_embedding=query_embedding[0],
            size=5
        )
        vector_time = time.time() - start
        results_summary.append({
            'method': 'Vector',
            'time': vector_time,
            'results': vector_results
        })
    except Exception as e:
        results_summary.append({
            'method': 'Vector',
            'time': None,
            'results': None
        })
# Test vector search
start = time.time()
try:
    query_embedding = await embeddings_generator.generate_embeddings([query])
    if query_embedding:
        hybrid_results = client.search_chunks_hybrid(
            query=query,
            query_embedding=query_embedding[0],
            size=5
        )
        hybrid_time = time.time() - start
        results_summary.append({
            'method': 'Hybrid',
            'time': hybrid_time,
            'results': hybrid_results.get("total", 0)
        })
    except Exception as e:
        results_summary.append({
            'method': 'Hybrid',
            'time': None,
            'results': None
        })

# Test hybrid search
start = time.time()
try:
    if query_embeeding:
        hybrid_results = client.search_chunks_hybrid(
            query=query,
            query_embedding=query_embedding[0],
            size=5
        )
        hybrid_time = time.time() - start
        results_summary.append({
            'method': 'Hybrid',
            'time': hybrid_time,
            'results': hybrid_results.get("total", 0)
        })
    except Exception as e:
        results_summary.append({
            'method': 'Hybrid',
            'time': None,
            'results': None
        })
# Display results
print(f"{'Method':<10} {'Time (s)':<10} {'Results':<10}")
print("-" * 30)
for result in results_summary:
    print(f"{result['method']:<10} {result['time']:<10.3f} {result['results']:<10}")

print("\nRecommendations:")
print("• BM25: Best for exact keyword matching")
print("• Vector: Best for semantic similarity")
print("• Hybrid: Best overall accuracy")

